In [32]:
# Lancement du modele

import torch
import torch.nn as nn
import torch.nn.functional as F


def get_device():
    """Renvoie le meilleur processeur disponible : cuda -> mps -> cpu."""
    if torch.cuda.is_available():
        return torch.device("cuda")      # GPU NVIDIA (Colab, cloud)
    if torch.backends.mps.is_available():
        return torch.device("mps")       # puce Apple Silicon (Mac M1-M4)
    return torch.device("cpu")           # sinon, le processeur classique


device = get_device()
torch.manual_seed(42)                    # même hasard pour tous : résultats reproductibles
print(f"PyTorch {torch.__version__} · calcul sur : {device}")
# Le chargement du corpus 
with open("fables.txt", "r", encoding="utf-8") as f:
    corpus = f.read()
    

# Ce code sert a décomposer chaque caractère du corpus en une seule fois. Par exemple, bonjour tout le monde sera décomposé en b, o, n, j, o, u, r,  , t, o, u, t,  , l, e,  , m, o, n, d, e. et les caratères seront triés en ordre alphabétique et n'apparaitront qu'une seule fois.
chars = sorted(set(corpus)) 
# print(chars)
vocab_size = len(chars)

# On crée un dictionnaire qui associe chaque caractère à un entier unique. Cela permet de convertir les caractères en entiers pour les utiliser dans le modèle. La fonction enumerate() est utilisée pour générer un index unique pour chaque caractère dans la liste chars. Le dictionnaire stoi (string to integer) est créé en utilisant une compréhension de dictionnaire, où chaque caractère c est associé à son index i.
stoi = {c: i for i, c in enumerate(chars)}

# On crée un dictionnaire qui associe chaque entier unique à son caractère correspondant. Cela permet de convertir les entiers en caractères pour interpréter les résultats du modèle. La compréhension de dictionnaire est utilisée pour inverser le dictionnaire stoi, en associant chaque index i à son caractère c.
itos = {i: c for c, i in stoi.items()} 

# On convertit le corpus en une liste d'entiers en utilisant le dictionnaire stoi. Chaque caractère du corpus est remplacé par son entier correspondant. La liste d'entiers est ensuite convertie en un tenseur PyTorch pour être utilisée dans le modèle.
data = torch.tensor([stoi[c] for c in corpus])

block_size = 16

class MiniLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.table = nn.Embedding(vocab_size, 24)
        self.reseau = nn.Sequential(
            nn.Linear(block_size * 24, 192),
            nn.Tanh(),
            nn.Linear(192, vocab_size),
        )
    def forward(self, x):
        return self.reseau(self.table(x).flatten(1))
    
    
model = MiniLM().to(device)

PyTorch 2.13.0 · calcul sur : mps


In [33]:
# Les fonctions de génération et d'entrainement du modèle.

def generer(prompt="\n", longueur=300, temperature=1.0):
    ctx = ([stoi["\n"]] * block_size + [stoi[c] for c in prompt])[-block_size:]
    sortie = []
    for _ in range(longueur):
        scores = model(torch.tensor([ctx], device=device))
        probas = F.softmax(scores / temperature, dim=-1)
        i = torch.multinomial(probas, num_samples=1).item()
        sortie.append(itos[i])
        ctx = ctx[1:] + [i]
    return prompt + "".join(sortie)

# Boucle pour l'entrainement du modèle
optimiseur = torch.optim.AdamW(model.parameters(), lr=1e-3)
for step in range(8001):
    ix = torch.randint(0, len(data) - block_size - 1, (64,))
    x = torch.stack([data[i:i+block_size] for i in ix]).to(device)
    y = data[ix + block_size].to(device)
    loss = F.cross_entropy(model(x), y)
    optimiseur.zero_grad()
    loss.backward()
    optimiseur.step()

In [34]:
print(generer("Quelle est la capital de la France?", longueur=300, temperature=.3))

Quelle est la capital de la France? le d'ant au son bour,
Cap lain que le corpage.
Sent mant que los ane,
Un e parte chos d'ant que la taux en repond mon ditise un bien ants qui moint des moi de arore avait couplie.
Le rence n'aux de furté,
Il fait pour se fait pes de serdeux de tros de coure un vour ca taise éraitre aut la teur de m
